# Глава 11. Агенты

## Введение
Языковой агент (LLM agent) — это система на базе большой языковой модели, которая самостоятельно управляет процессом решения задачи: разбивает её на шаги, выбирает и вызывает инструменты, анализирует полученные результаты и на их основе определяет следующие действия, пока не достигнет цели. Отличие от обычного LLM-приложения — наличие цикла автономного принятия решений: планирование → действие → наблюдение → следующее действие.

<img src="img/agent.jpg" width=500>

## Рассуждающие модели
Ранние языковые модели работали по принципу "вопрос - ответ", пользователь формилировал задачу в виде входного промпта, а модель генерировала наиболее вероятное продолжение. Если на простых запросах этого было достаточно, то на задачах, требующих многошагового логического рассуждения, модели часто ошибались, пытаясь сразу «перепрыгнуть» к ответу. Оказалось, что дело не столько в нехватке знаний, сколько в отсутствии у модели некоторого «черновика»: у нее просто не было пространства, где она могла бы развернуть ход своих мыслей. __Рассуждение__ (Reasoning) - это способность модели фиксировать *промежуточные* шаги своего логического вывода перед генерацией финального ответа.

В 2022 году [[Wei et al., 2022]](https://arxiv.org/abs/2201.11903) из Google обнаружили, что если попросить модель не выдавать ответ сразу, а сначала проговаривать промежуточные шаги, качество её ответов на сложных логических задачах резко растёт. Они это делали через Few-shot промптинг: модели явно показывали пример похожей задачи и вот, смотри как она рассуждает, нужно так же! Этот вид промптинга назвали **Chain-of-Thought (CoT)** размышлением.

Чуть позже ([Kojima et al., 2022](https://arxiv.org/abs/2205.11916)) показали, что для аналогичного результата даже не нужно добавлять в промпт примеры. Достаточно одной фразы вроде «Рассуждай шаг за шагом», чтобы запустить тот же самый эффект. Удивительно, но добавление одной фразы увеличивает точность ответов модели в 2-4 раза при решении сложных логических задач, а с примерами прирост ещё выше. Естествено, такое дешевое улучшение запустило целую волну рассжудающих моделей. Обратная сторона - стали бороться с избыточным потреблением токенов, поскольку рассуждение существенно увеличивало время ответа.

Рассуждение по сути предоставляет модели пустой мольберт, пространство, в котором сложная задача может решаться поэтапно.

Тут стоит отметить, что CoT помогает только там, где есть, что декомпозировать (арифметика, логика, многоступенчатые выводы). На обычных задачах он либо бесполезен, либо даже вредит, так как лишние рассуждения добавляют шум и возможности ошибиться. Тестируют рассуждения чаще всего на математике, поскольку математиеские задачи лучше всего подходят под определение задач, требующих многошагового решения, там есть и строгое поэтапное ветвление логики и один объективный конечный ответ.

В 2022 году [(Zhang et al)](https://arxiv.org/pdf/2210.03493) из Amazon пытались найти компромисс между тривиальностью zero-shot подхода ("Размышляй вслух") и качеством few-shot подхода ("Вот примеры, как ты должна рассуждать"). Свой метод они назвали __Auto-CoT__ и по сути он стал одним из первых механизмов генерации синтетических примеров для языковых моделей (в оригинальной работе с рассуждениями, а в дальнейшем с любыми инструкциями). Малые модели плохо приспособлены к рассуждению. Но их можно частично "запатчить", добавив примеры хороших рассуждений в промпт, либо полноценно дообучив. Но вручную собирать датасеты с рассуждениями очень дорого. Поэтому обращаются к большим моделям, которые просят синтетически нагенерировать примеры рассуждений. По аналогии с тем, как кафедры в университете привлекают аспирантов к ведению занятий у первокурсников, для чего те придумывают свои задания. Суть метода в том, что модель берёт собранную базу вопросов, кластеризует их по типу, выбирает по одному характерному вопросу каждого типа и отвечает на них с рассуждением. Множество таких генераций обхединяют в промпт, который можно передавать на вход малой модели.

Далее стало понятно, что одна цепочка рассуждений ненадёжна, ведь модель может пойти в своем рассуждении по ошибочному пути, и чем длиннее рассуждение, тем с большей вероятностью она это сделает. ([Wang et al., 2022](https://arxiv.org/abs/2203.11171)) предложили генерировать не одну цепочку рассуждений, а сразу набор. Температура генерации беспечивала разнообразие. После того, как набор рассуждений получен, можно выбрать наиболее перспективное. Чем чаще траектории сходятся к одному ответу (являются констистентными), тем вероятнее он правильный, тогда как ошибки обычно уникальны. Такой подход назвали само-консистентным __Self-Consistency__ рассуждением.

([Yao et al., 2023](https://arxiv.org/abs/2305.10601)) развили эту идею и переосмыслили рассуждение как поиск по дереву. Вместо линейной цепочки модель на каждом шаге порождает несколько вариантов продолжения рассуждения, оценивает их перспективность, углубляется в удачные ветви и откатывается от тупиковых. Модель получила название __Tree-of-thoughs__.

<img src="img/cot4.png" width=500>

Чтобы реализовать подбную логику, нужно явно вести понятие шага рассуждения - некоторое количество текста, описывающего одно логическое действие. Здесь не переусложняют, в промпте просто явно прописывают "Выполни ровно один логический шаг". Так модель понимает, когда ей нужно остановиться для переоценки траектории.

Далее цепочки нужно как-то оценивать, не только в конце, но и по ходу генерации. Кроме того постоянное ветвление влечет за собой экспоненциальный рост количества траекторий, которые надо проверять. Соотвественно нужно либо грамотно настроить критерий отбрасывания беперспектиных траекторий (если видно, что ветвь очевидно тупиковая). Либо использовать алгоритм Beam Search, когда в каждый момент генерации у нас отобрано не более топ K наиболее перспективных вариантов продолжений, а остальные отбрасываются.

В роли оценщика (__Verifier__ или Model-as-a-judge) обычно выступает та же самая модель, она отвечает на вопрос: похоже ли, что рассуждение приведет к правильному ответу? Траекторию можно оценивать двумя способами. Первый способ - в абсолютных значениях, например по шкале ("sure" - да, уже пришли к ответу, "impossible" - ветка рассуждения плохая ни к чему не приведет, "maybe" - пока нельзя сказать). Второй ранжированием, оцени потенциал рассуждения и выбери топ-5 перспективных цепочек.

К минусам самопроверки можно отнести то, что модели часто предвзяты (Self-Correction Fallacy). Искуственный интеллект традиционно плохо калибрует свою уверенность (но тут и у людей с этим не очень). Да и его склонность "конформизму" никуда не девается - если контексте есть утверждение X, модели проще ему следовать, чем опровергать.

Единого рецепта, как с этим бороться нет, но есть набор рекомендаций. Дублировать можно не только генерации, но и шаги оценки, причем желательно их делать разными модлями. От команды случайно отобранных присяжных ожидаешь более справедливой оценки, чем от одного судьи. Кроме того не забывать грамотно формулировать промпты на проверку, не "как ты оцениваешь рассуждение?", а просить проделать аналитическубю работу, провести тесты и сравнения. В современных моделях используется Reinforcement Learning [подробнее].

[(Besta et al)](https://arxiv.org/abs/2308.09687) развивают мысль ещё дальше. А что если дать модели возможность не только ветвить, но и объединять разные траектории. Тогда вместо дерева (Tree) рассуждений мы получим ациклический граф рассуждений (DAG, Direct Acyclic Graph). Такой вариант назвали __Graph-of-thoughts__. Идея в том, что перспективные траектории могут дать синергию при объединении [пример].

Ниже мы увидим, что часто модели востребованы там, где нужно выполнять некоторые действия, например, писать код выполняемой программы. Вместо плана пишется код. Так сделали авторы __Program-of-Thoughts__.

### Масштабирование на инференсе
Всё, что мы рассмотрели выше - пример того, что можно объединить зонтичным теримном __Test-Time Scaling__. Идея в том, что качество ответа языковой модели растёт, если позволить модели «думать» дольше.

Долгое время в индустрии был популярен сформулированный Ричардом Саттон принцип, суть которого сводилась к тому, что единственный надежный способ сделать нейросеть умнее — это наращивать её обучение, через рост количества параметров модели, через увеличение обучающей выборки, через увеличение времени обучения. Благо, закон Мура в то время это позволял. Но уже к 2024 году стало понятно, что мы упираемся в лимит по возможностям роста. Каждый следующий миллиард параметров стоил дороже, а процентов качества добавлял всё меньше. И в индустрии стали активнее искать другие пути.

В 2024 году [(Snell et al)](https://aclanthology.org/2025.acl-long.232.pdf) из Google DeepMind обратили внимание на успех CoT и ToT и решили проверить, насколько это универсальный результат. Они взяли простую модель и прогнали её в двух режимах - с коротким рассуждением по модели Self-Consistency и с длительной генерацией в парадигме Tree-of-thought. В реузльтате сделали несколько важных выводов. Во-первых, они показали, что улучшение зависит от сложности задачи - слишком долгое рассуждение над простой задачей наоборот ухудшает качество. Во-вторых, вывели формальный закон масштабирования, согласно которому малая модель, которой дали достаточно подумать, может превосходить аж в 14 раз более крупную модель. Это проиллюстрировало важность инференса, как этапа работы модели, и породило целое направление исследований. Иными словами, вместо зубрежки учебников и прорешивания примеров в качестве метода подготовки к экзамену, стали чаще полагаться на свой интеллект - "на экзамене будет время, придумаю".

Так закрепился термин Test-Time Scaling. В литературе подобную смену парадигмы часто иллюстрируют как переход от "системы 1" к "системе 2" в терминах когнитивной психологии Даниеля Канемана. Система 1 это быстрый интуитивный интеллект, выдающий ответ сразу и почти без усилий. А система 2 наоборот медленное осознанное размышление, требующее взвешивания аргументов и проверки промежуточных шагов. Такую параллель проводил в частности сам Сэм Альтман, говоря о только что вышедшей модели OpenAI o1, где впервые среди коммерческих моделей появился навык рассуждения. Чтобы подчеркнуть значимость смены парадигмы, OpenAI даже обнулили нумерацию своей линейки моделей.

В идеальном сценарии рассуждающая модель сама определяет, когда ей закончить думать: она ставит тег </think> и выдает ответ. Но на практике ИИ-инженерам часто приходится управлять этим процессом. В 2024 году [(Muennighoff et al, 2024)](https://arxiv.org/abs/2501.19393) из Стенфорда решили протестировать механизм, который назвали __Budget Forcing__. В рамках своей модели __s1__ (simple test-time scaling) они решили повторить логику o1, но не разрабатывать сложные RL инструменты, а действовать "в лоб": модели выделяется жестко определенное время на размышление, которое она должна соблюдать. С одной стороны оно мотивирует модель не лениться думать (Underthinking), с другой - не уходить в бесконченый цикл размышления (Overthinking), чтобы модель не "намудрила" на простой задаче.

Ограничение снизу можно реализовать через промпт («Твоё рассуждение должно содержать не менее 15 независимых шагов проверки. Если ты уверена в ответе, найди три причины, почему ты можешь ошибаться»). Либо через искусственное занижение вероятности токенов \</think\> или [EOS]. В начале генрации вероятность небольшая, затем ближе к лимиту времени растёт до 1. Аналогия из жизни - вы сделали задание раньше выделенного времени и уверены в ответе. Вместо того, чтобы сдавать работу, выгодно несколько раз её перепроверить, вы так увеличиваете вероятность найти ошибку.

Ограничение сверху можно реализовать через мониторинг генерации токенов <think>. Как только лимит исчерпан, система принудительно добавляет в контекст токен </think> и требует генерировать финальный ответ. «Время вышло, сдаем листочки».

### Как сделано в DeepSeek-R1
В сентябре 2024 OpenAI выпустила свою модель o1, которая стала первой крупной моделью для выполнения сложных многошаговых рассуждений и обученной с помощью подкрепления (RL). На тестах модель достигла уровня лучших выпускников школ, что было прорывом. Но к сожалению она оставалась закрытой и проприетарной.

В начале 2025 года китайский стартап DeepSeek выпустил новую версию своей одноименной модели, которая также была заточена под рассуждение [DeepSeek-R1, 2025](https://arxiv.org/abs/2501.12948). Но в отиличие от OpenAI, они сделали модель открытой, а также опубликовали техническое описание процесса обучния.

Появление модели DeepSeek-R1 стало важной вехой, так как она не только догнала (а в некоторых аспектах превзошла) o1 по тестам, но и сделала технологии доступными для всего исследовательского сообщества, что дало толчок развитию области рассуждений. Успех модели так впечатлил инвесторов (особенно заявленная стоимость дообучения в 300 тысяч долларов, которая была в разы ниже OpenAI), что в моменте новая модель обрушил акции NVidia, главного производителя дорогостоящих чипов, на 20%. Позднее правда стоимость откатилась назад.

Но самое главное, что показала модель, что выдающиеся способности к рассуждению могут возникнуть спонтанно в процессе обучения с подкреплением, без необходимости какого-либо дорогостоящего обучения на примерах. Одним из поразительных эффектов был так называемый "A-ha moment", когда в процессе размышления модель в какой-то момоент буквально говорит "подождите-ка... я поняла" , понимает, как правильно решить задачу, зачеркивает все написанное ранее и пишет правильное решение.

СМИ назвали это проявлением осознанности и подавали как способность искусственного интеллекта к саморефлексии, хотя научное сообщество сошлось, что антропоморфная трактовка здесь не совсем корректна: это хоть и впечатляющая, но просто иллюзия когнитивного процесса. Это пример эмерджентности - свойства появлению у модели новых способностей, которые не закладывались изначально и проявляются по мере роста сложности системы.

## Другие свойства агентности

## Реактивность
Мы научились масштабировать рассуждение, теперь мы можем бить его на отдельные шаги и даже следовать траектории мысли. Но рассуждение, замкнутое внутри модели - это монолог. Такая модель - это всё та же модель, работающая по принципу черного ящика, просто тратящая больше токенов на этапе генерации. Нам же хочется, чтобы рассуждение было менее линейным, и более интеракивным - могло и влиять, и реагировать на внешнюю среду (в нашем случае это соотвественно контекст). Свойство модели подстраиваться под изменение контекста будем называть реактивностью.

([Yao et al., 2022](https://arxiv.org/abs/2210.03629)) из Google описали паттерн рассуждения **ReAct**, который реализовывал ровно такой подход: чередовал рассуждение и действие. Название ReAct - склеено из *Reasoning* и *Acting*. Алгоритм рассуждения состоит из следующих шагов:
1) модель рассуждает («чтобы ответить, мне нужно узнать X»)
2) выбирает дальнейшее действие (например, поиск)
3) получает наблюдение (результат поиска)
4) возвращается к 1: повторяет рассуждение, но уже с учётом нового факта
5) генерирует финальный ответ

В качестве модели использовалась обычная языковая модель, а обучение делали в zero-shot режиме: через показ примеров. Реактивность реализовали на примере обращения к базе знаний (Википедии). Чтобы модель могла обогащать свой контекст, ей дали доступ к трём функциям: подгрузке информации по ключевому слову: "[seach entity]", поиску строки в документе "[lookup string]", и финалзиации ответа "[finish answer]". Здесь мы видим один из первых примеров использования инструментов, а также прототип Retrieval Augmented Generation систем, которые позже станут стандартом.

Как и в случае с рассуждением, реактивность важна не всех задачах, а только там, где внутренних знаний модели не достаточно и нужно обогащение внешней информацией.

## Использование инструментов
У людей первые механические орудия (каменное рубило) появились около 3 млн лет назад. Они фактически заменили человеку зубы, открыв доступ к высококалорийной пище. предположительно главной предпосылкой к их появлению стало развитие прямоходждения (Homo Erectus вышли из леса), развитие кистей, и появление абстрактоного мышления (орудие нужно еще изготовить). А высокая социализированность по сравнению с другими видами позволила поствить это умение "на конвейер". В результате вектор эволюции сместился: вместо статического развития у вида Homo появилась возможность изменять окружающую среду.

В контексте языковых моделей инструмент (Tool) - это любая внешняя по отношению к модели возможность изменить контекст. Самые первые инструментов: поиск (Web или база данных), калькулятор, прогноз погоды и прочее. Как правило инструменты параметризованные - запрос

В целом паттерн ReAct уже описал, как можно рассуждение действиями, но не хватало универсального механизма, который описал бы, как подключать к модели произвольные инструменты. ([Schick et al., 2023](https://arxiv.org/abs/2302.04761)) решили восполгнить это упущение и масштабировали подход ReAct, перейдя от zero-shot к обучению. Они назвали свою модель **Toolformer**, которая на какое-то время стала стандартом применения инструментов.

Обучение реализуется в два шага. На первом модели показывается несколько few-shot примеров, как можно использовать инструмент и та пытается вставлять везде в тексте, где это выглядит разумным. Далее она смотрит на внутренние метрики (как правило, перплексию), и если вставка ощутимо растит метрику справа от неё, значит здесь инструмент полезен - и вставка помечается, как положительный пример.

На втором шаге языковая модель полноценно обучается по сгенерированному на предыдущем шаге синтетическому датасету. В целях экономии авторы не стали делать бесконечный цикл, как в ReAct, ограничились разовым обогащением.

<img src="img/toolformer.png" width=750>

RL для рассуждений:
- __STaR__<br>Self-Taught Reasoner (Zelikman et al., 2022) — берём задачу с однозначным ответом. Генерируем одну траекторию жадным декодированием. Если ответ верный — в датасет. Если нет — переспрашиваем, подав правильный ответ в промпт, получаем траекторию задним числом, кладём в датасет уже без подсказки. Собираем SFT-датасет, дообучаем базовую модель (не предыдущую итерацию). Повторяем.
- __ReST__<br>(Gulcehre et al., 2023; Singh et al., 2023) — Берем задачу с однозначным ответом. Генерируем N траекторий (разнообразие обеспечивается температурой генерации). Оставляем приведщие к праильномк ответу, из них собираем SFT датасет и дообучаем на нем модель
- __RFT__<br>Метод Rejection Sampling Fine-Tuning, (Yuan et al., 2023) — то же, но сэмплируем много вариантов и фильтруем по проверяемому критерию
- __RAFT__<br>(Reward-rAnked FineTuning, Dong et al., 2023)
- Исторический предок в RL-литературе: expert iteration (Anthony et al., 2017, та же идея испольщуется в AlphaZero)

## Планирование
Для коротких задач простого последовательного рассуждения часто бывает достаточно. Но иногда задача слишком масштабна и нужен специальный механизм её координации. В роли такого механихма выступает планирование. Если рассуждение - это просто поток мысли, то план - это а) декомпозиция задачи на подзадачи б) чёткое определение зависимостей и порядка выполнения в) критерий завершения: по чему мы поймём, что подзадача действительно решена.

Планирование полезно в двух случаях. Когда ландшафт решения слишком сложен - задача поставлена слишком абстрактно или наоборот задано слишком много конкурирующих требований. Либо когда высок риск неправильных действий и сменить траекторию решения находу непросто.

В обоих случаях выгоднее продумать ход решения заранее. Пример такой задачи - оформление командировки сотрудника из Москвы в Мехико на конкретные даты в соотвествии с выделенным бюджетом. У сотрудника есть две банковские карты: одна заблокирована для международных операций, вторая — с лимитом в 50 тысяч рублей. Агент при этом умеет через API проверять цены, искать визовые требования к каждой стране, может проверять остаток на карте.

Планирование желательно оформить как отдельный шаг пайплайна и разнести его с выполнением. Тут две идеи: либо сначала составляем план, а потом его выполняем, либо циклически корректировать план на ходу выполнения. Первый подход проще и экономичнее, так как достаточно всего одной итерации планирования, но вместе с тем выше риск ошибки, если план окажется плохим. Второй подход гибче, но выше риск погрязнуть в корректировках. Тут уместна аналогия с waterfall и agile методологиями разработки, логика примерно та же. Ниже рассмоторим примеры методов каждой из двух групп.

### Разовое планировние
Самый простой пример единоразового планирования - это паттерн __Plan-and-Solve__. [(Wang et al., 2023)](https://arxiv.org/abs/2305.04091)) по аналогии с Chain-of-thought просто попросили модель в промпте "При решении задачи сначала построй план, а потом его выполняй". Уже этого было достаточно, чтобы получить более хорошие решения на задачах, требующих планирования.

Имея список подзадач, их можно запускать на выполнение параллельно, что делает генерацию более эффективной. К этому же классу можно отнести модели __Least-to-most__, которая ранжирует подзадачи по сложности.

Модель __ReWOO__ (Reasoning without observation) реализует принцип "планируй без данных". Перед выполнением разово генерируется общий план. Каждый пункт плана - это вызов инструмента, причем в параметре он может передавать результат выполнения предыдущих. Далее компонент Worker вызывает инструменты в соответствии с построенным графом зависимостей и заполняет плейсхолдеры. Наконец, модель видит общий план с заполненными данными и генерирует финальный ответ.

[(Shen et al, 2023)](https://arxiv.org/abs/2303.17580) из Microsoft в своей модели __HuggingGPT__ рассматривают репозиторий моделей HuggingFace как набор инструментов. На вход модель принимает промпт на естественном языке, например, "Посмотри на фотографию здания, определи, что это за здание, и расскажи его историю". Затем происхоит планирование, модель декомпозирует задачу на связанные подзадачи. На шаге Model Selection выбирает для каждой подзадачи наиболее подходящую физическую модель (просто по базе текстовых описаний и метаданным). Затем выбранные модели применяются в исполняемой серде согласно построенному графу зависимостей. 

Это пример раннего мультимодального агента, который реализует мультимодальность не нативно, а с помощью внешних инструментов. Сами авторы предложили назвать такую стратегию "языковая модель как орекстратор" (LLM-as-a-controller). Она составляет план, выполнение на внешних моделях.

Исполнитель не обязательно языковая модель (сама же, как в ReWOO или другая как в HuggingGPT), в его роли может выступать и внешний инструмент. Такую логику реализует, например, модель __LLM+P__ [(Liu et al., 2023)](https://arxiv.org/abs/2304.11477), в рамках которой план составляется не на естественном языке, а согласно стандарту [PDDL](https://en.wikipedia.org/wiki/Planning_Domain_Definition_Language) (формальный язык описания задачи). Далее внешний планировщик задачу решает и переводит в план. Далее этот план выполняется, и опционально генерируется обратно в человекочитаемый формат. 

Похожий принцип в модели __ProgPrompt__. В промпт добавляется описание задачи на естественном языке, а также список сигнатур всех доступных функций. Языковая модель генерирует код и он выполняется в подготовленной исполняемой среде.

### Итеративное планирование
Методы этой группы напоминают расмотренный выше цикл ReAct, но здесь к рассуждению добавляется ещё шаг планирования. На каждой итерации планируется и выполняется ровно один шаг, после этого происходит переоценка результатов и цикл повторяется.

Модель __Self-Ask__ от [(Press et al., 2022)](https://arxiv.org/abs/2210.03350) минимально отходит от классического рассуждения и повторяет цикл React. Единственное отличается тем, как выбирается шаг генерации. После каждого шага модель формулирует некий уточняющий вопрос, отвечает на него (сама или выполняет поисковый запрос) и продолжает на следующем шаге.

Кроме того авторы ввели понятие разрыва композиционности (compositionality gap) — это свойство модели, зная ответы на отдельные подвопросы  ошибаться в финальном ответе. Это классическая ситуация "Назвал буквы и не смог назвать слово". Причем с ростом размера модели этот разрыв не сокращается: комбинировать факты сложнее, чем запоминать. И дейстивтельно, они показали, что уточняющие вопросы в Self-Ask уменьшают этот gap. И это, пожалуй, главный аргумент в пользу явного планирования.

Ровно та же идея в __Successive Prompting__ [(Dua et al., 2022)](https://arxiv.org/abs/2212.04092), но там разносят уточняющий вопрос и ответ на него по разным моделям. Это позволяет гибче настраивать систему.

Выше мы отмечали, что излишнее усложнение инференса может портить картину. С планирование то же самое, в своей модели __ADaPT__ [(Prasad et al., 2023)](https://arxiv.org/abs/2311.05772) предложили декомпозировать задачу не всегда, а только там, где это неоходимо. Агент сначала пробует выполнить задачу целиком. Если не справился, именно эта подзадача рекурсивно разбивается, и процедура повторяется. Получается, что уровень детализации плана подстраивается сразу под две вещи — сложность задачи и способности конкретного исполнителя.

Более общая схема — __Plan-and-execute с перепланированием__. Выполнили шаг, сравнили результат с ожиданием, при расхождении переписали остаток плана.

Отдельно стоит __RAP__ (Reasoning via Planning, [Hao et al., 2023](https://arxiv.org/abs/2305.14992)). В выполняет действие потом реактивно на него реагирует. А что, если модель будет предсказывать результат до выполнения? Поиск ведётся методом Монте-Карло по дереву (MCTS) в четыре фазы: *отбор* — спуск по дереву по критерию, балансирующему эксплуатацию известного и исследование нового; *расширение* — модель порождает кандидатов-действий в выбранном узле; *симуляция* — прогон до терминального состояния с оценкой награды (самооценка модели, вероятность действия, доменная эвристика); *обратное распространение* — обновление оценок вверх по дереву.

__PlanBench__ [(Valmeekam et al., 2023)](https://arxiv.org/abs/2206.10498) взял стандартные домены из соревнований по автоматическому планированию (Blocksworld, Logistics) и проверил модели на восьми аспектах планирования, от генерации плана до рассуждения о последствиях. Самая показательная часть — «затемнённые» домены (Mystery Blocksworld), где осмысленные названия действий и объектов заменены на бессмысленные строки. Модели на затемнённой версии рушатся почти до нуля. Это довольно сильный аргумент, что часть успеха объяснялась узнаванием знакомого домена по обучающим данным, а не планированием. Рассуждающие модели поколения o1 подняли планку заметно, но вместе с ней и стоимость вывода.

В целом планирование — опциональный элемент языкового агента, и, как и рассуждение, оно окупается только при достаточной сложности задачи. На практике его чаще всего видно в режимах глубокого исследования у чат-ботов и в кодовых агентах вроде Claude Code или Codex, где задача требует десятков шагов и следования выбранной траектории. Обзор методов с более подробной таксономией — [(Huang et al., 2024)](https://arxiv.org/abs/2402.02716).

## Память
Языковые модели не хранят состояния между вызовами, взаимодейтсвие реализуется через контекст, который каждый раз читается полностью заново (а KV кэш?). Но контекстное окно конечно. При всех оптимизациях не бывает больше 1M токенов. Это фундаментальное ограничение: без памяти агент не может вести длинную задачу или помнить о прошлых взаимодействиях

В контексте языковых моделей выделяют следующие виды памяти:

Краткосрочная память (Short-term Memory, STM) содержит информация в текущем контекстном окне. Это в частности, история последних сообщений в чате. Она быстрая, но ограничена максимальным размером контекста, обычно от 4k до 200k токенов.

Долгосрочная память (Long-term Memory, LTM) содержит информацию, которая может быть использована в будущих сеансах. Физически хранится в базах данных, векторных хранилищах или файлах.

Эпизодическая память (Episodic Memory) хранит воспоминания о конкретных событиях и опыте (например, "Во вторник пользователь попросил перенести демонстрацию").

Семантическая память (Semantic Memory) хранит факты и общие знания, независимые от времени и контекста (например, "Пользователь предпочитает встречи во второй половине дня").

Процедурная память (Procedural Memory): Хранит информацию о том, как выполнять задачи — инструкции, рабочие процессы и правила использования инструментов.

Рабочая память (Working Memory): Подмножество информации, которая в данный момент загружена в контекстное окно и доступна для немедленного использования.

[(Liu et al., 2023)](arXiv:2307.03172) изучили, как именно используется контекстная память, оказалось, что эффективность доступа крайне неравномерна. Этот эффект получил название «Lost in the Middle».

В рекуррентных сетях память - это агрегат контекста. 

Базовые приёмы 
- суммаризация истории (сжать прошлый диалог, чтобы он влез в окно),
- векторные хранилища (сохранять факты и доставать релевантные по семантическому поиску — основа RAG)
- аккуратное управление контекстным окном

([Packer et al., 2023](https://arxiv.org/abs/2310.08560)) вдохновились тем, как организована память в ОС и предложили **MemGPT** одну из первых моделей, реализующих механизм внешней памяти. В ней ящыковая модель реализует двухуровневную модель памяти (оперативная vs постоянная) и сама решает, что держать в «оперативном» контексте, а что вытеснить во внешнее хранилище и подгрузить обратно при необходимости. Можно провести аналогию с [виртуальной памятью](https://en.wikipedia.org/wiki/Virtual_memory) и страничной подкачкой (page swap).

<img src="img/memgpt1.png" width=500>

Наличие памяти позволяет работать с задачами длиннее, чем его контекстное окно.

## Рефлексия и самообучение
Важное свойство языковых агентов - способность критиковать свои решения и улучшать их, не дожидаясь внешнего стимула.  Если планирование, которое мы рассматривали выше, это активность перед генерацией, то рефлексия - это активность после генерации ответа. Цель рефлексии - повторно запустить генерацию с исправлением. Проводя аналогию с программной разработкой, планировние - это выбор задач из бэклога перед началом спринта, а рефлексия это ретроспектива в конце спринта.

(Madaan et al., 2023) описали вероятно самое простое решение, после генерации ответа запустить еще один цикл с промптом "покритикуй свое решение". Подход назвали __Self-Refine__. Он хорошо подходит для например, оценки стиля. Но для проверки фактичности не подходит.

Проблема с саморефлексией - если модель оценивает сама себя, она предвзята. Распределения идентичны.

Частично можно исправить, попросив модель оценить не смотри на оригинальный ответ. Этот подход назвали __Chain-of-Verification__. Идея в том, что сначала модель генерирует черновик ответа, затем по этому черновику формулирует набор проверочных вопросов и получает ответы на них. После этого компонует ответ.

Пример : «Назови нескольких политиков, родившихся в Нью-Йорке»
Черновик: Хиллари Клинтон, Дональд Трамп, Майкл Блумберг.
Сгенерированные вопросы:
1. Где родилась Хиллари Клинтон?
2. Где родился Дональд Трамп?
3. Где родился Майкл Блумберг?
Ответы (каждый — отдельный вызов, черновика в контексте нет):
1. Чикаго, Иллинойс          ← расхождение
2. Куинс, Нью-Йорк           ← подтверждено
3. Бостон, Массачусетс       ← расхождение

От проблемы идентичных распределений можно частично, если разнести исполнителя и верификатор по разным моделям. ([Shinn et al., 2023](https://arxiv.org/abs/2303.11366)) предложили добавлять критику промптом. Модель получила название **Reflexion**. Дает +11%

тесты / компиляция
CRITIC - проверяет ответ фактовым запросом
отдельная LLM оценивает

<img src="img/reflection2.png" width=300>

**Voyager** ([Wang et al., 2023](https://arxiv.org/abs/2305.16291))<Br>довёл идею до пожизненного обучения на примере агента в Minecraft. Три механизма работают вместе: автоматический учебный план (агент сам ставит себе посильно усложняющиеся цели), библиотека навыков (удачные решения сохраняются как переиспользуемый код и комбинируются в более сложные) и обратная связь от среды-песочницы. Так агент постепенно накапливает компетенцию, а не решает каждую задачу с нуля

Агент становится самоулучшающимся в пределах сессии или «жизни», а библиотека навыков превращает разовые решения в композиционную, переиспользуемую компетенцию


<img1 src="img/agent_papers.jpg" width=500>




# Мультиагентные системы
Если агенты стали настолько умными, что способны автономно решать заадчи, то почему бы не запустить сразу команду агентов? Ведь на бытовом уровне, чтобы запустить амбициозный проект, нужна команда хороших специалистов. Как бы хороши они не были по отдельности, их навыков просто не хватит. Один агент может перегружается задачами, путать роли (если их много), терять контроль над длинным процессом. __Мультиагентная система__ - множетсво языковых агентов, работающих вместе в координации для выполнения глобальной цели

Аргументы в пользу использования команды агентов:
- способ прочитать больше, чем влезает в контекст
- можно распараллелить выполнение
- специализация агентов:
    - безопасность - агент выполняет строго определенный набор действий
    - экономия - для простых задач требуется меньше токенов
    - изоляция ошибок - агент можно перезапустить

Независимость однако порождает разногласие и чем больше в системе агентов, тем больше нужно уделять внимания их координации. В целом системы, ориентированные на чтение проще, чем системы ориентированные на запись (Cognition vs Antrophic). 

Можно предположить, что много агентов улучшает "широту" мышления (Divergent Thinking) - якобыкоманда агентов может предложить больше вариантов решения, чем отдельный агент. Но на практике проихсодит скорее наоборот. [(Wang et al, 2024)](https://arxiv.org/abs/2406.06461) показали, что чем больше координируют свое рассуждение, пытаясь прийти к ответу, тем сильнее падает энтропия (разнообразие) ответов. Это происходит из-за того, что . Для сравнения в методе рассуждения self-consistency такого не происходит

Другой аргемент - повышает точность ответов (improves factuality & reasoning). Это действительно так, но незщначитеолтьнро, тот же Chain-of-thought достигает такого же результата дешевле

Что лучше валидирует ответ (Validation)

Мультиагентые системы - один из главных трендов развития ИИ начиная с 2025 года. Стоит отметить, что более поздние работы показали, что преимущество команды агентов перед индивидуальными исполнителями не так очевидно. Например, здесь [(Tran et al, 2026)](https://arxiv.org/pdf/2604.02460) показывали, что при фиксированном бюджете одноагентная конфигурация может быть ни чуть не хуже. Или здесь [(Xu et al, 2026)](https://arxiv.org/pdf/2601.12307) авторы исследуют важность узкой специализации и опровергают её, показав что ту же систему можно реализовать одной моделью, назначая ей роли промптами

Однако и риски растут. Появляются каскадные ошибки, становится возможным prompt injection, работа иногда зацикливается

Поэтому важна координация. Как определяется приоритеность команд? Например, приказ смежного агента противоречит системному промпту. Как определяется порядок выполнения в мультагентных системах: может жестко по графу. пример - игра в мафию
- агент сам определяет (push режим)<br>игра в мяч
- GroupChatManager - коорднатор<br>модератор на панельной сессии
- по условию / триггеру<br>

Агенты - это как правило отдельные сессии или микросервисы, выполняют работу параллельно, но важно отметить, что их координация дискретна, нужно должаться полного выполния перед следующим шагом

### AutoGen
Одна из первых работ, описавщая - фреймворк **AutoGen** от Microsoft. ([Wu et al., 2023](https://arxiv.org/abs/2308.08155)) предложили как можно организовать совместную работу множества агентов, просто дав им возможность коммуницировать на естественном языке

Работа популяризировала термины Conversable Agent (диалоговый агент) - модель, готовая коммуницировать с внешним миром через язык и Conversable Programming (диалоговое программирование) - способ координации работы между агентами при решении ими распределенной задачи, когда задачи ставятся текстом

В работе вводят несколько классов агентов по 
- AssistantAgent - "мозги" для генерации рассуждения
- UserProxyAgent - "руки" для автономного выполнения действий. Моделирует "что бы сделал пользователь"
- GroupChatManager - (опционально) координатор, выбирает, кто будет следующий действовать

<img src="img/autogen1.png" width=600>

Модель развитвается. В более поздней версии реализовали модель вычислений Actor-model

> [Actor Model](https://en.wikipedia.org/wiki/Actor_model) - это математическая модель асинхронных вычислений. Актор - это объект, который умеет а) принимать сообщения б) отправлять сообщения в) порождать дургих акторов г) выполнять какое-то действие<br>
Модель разрабатывали еще с 1973 года. На основе нее были реализованы многие инженерные фреймворки для разработки Message-driven applications, например, фреймворк [Akka](https://en.wikipedia.org/wiki/Akka_(toolkit)) для Java<br>
Похоже на концепцию микросервисов: вычисление тоже бъется на отдельные независимые куски, запускаемые асинхронно, но разница в уровне детализации. Akka работает на уровне отдельных объектов

### MetaGPT

В другой работе **MetaGPT** ([Hong et al., 2023](https://arxiv.org/abs/2308.00352)) принцип тот же, но тут разделение труда более стандартизированный. Там пошли от метафоры организации: каждый агент выполняет свою роль, как это бывает в софтверной компании: продакт-менеджер, архитектор, инженер

<img src="img/metagpt1.png" width=500>

### Оркестрация
По мере усложнения систем линейных цепочек становится мало — нужен переход к управляемым графам состояний. Графовая оркестрация (типичный представитель — [LangGraph](https://github.com/langchain-ai/langgraph)) описывает агента как граф: узлы — это шаги, рёбра — переходы, есть общее состояние и допустимы циклы. Это даёт явный контроль над тем, что и в каком порядке происходит.

Хороший словарь паттернов задаёт статья Anthropic [«Building Effective Agents»](https://www.anthropic.com/engineering/building-effective-agents). Она проводит важное различие между workflow (заранее заданные маршруты, по которым модель ведут жёстко) и собственно агентами (модель сама управляет своим процессом). И описывает базовые композиционные паттерны: chaining (последовательная цепочка), routing (маршрутизация запроса в нужную ветку), параллелизация, оркестратор-исполнители, оценщик-оптимизатор

### Стандарты подключения (MCP)
С ростом популярности использования инструментов, встал вопрос интеграции: каждый раз писать «переходник» между моделью и очередным сервисом дорого и с ростом их разнообразия затраты растут экспоненциально. Нужна стандартизация

[Model Context Protocol (MCP)](https://modelcontextprotocol.io) — это открытый протокол стандартизированного подключения LLM к инструментам и данным; его можно визуализировать как «USB порт для ИИ». 

Появляются серверы инструментов и источники данных, которые реализуют протокол один раз, после чего любой совместимый агент может ими пользоваться. Тема здесь не столько техническая, сколько системная: почему стандарт важен для масштаба — он развязывает интеграцию и приложение, и за счёт сетевого эффекта порождает целую экосистему переиспользуемых компонентов

Были попытки навязать войну стандартов: A2A от Google, но на текущий момент пока стнадартом ялвяется MCP

### Фреймворки
Инструмент нужно выбирать под задачу, а не привязываться к одному. Полезно держать в голове грубую карту:
- LangGraph - про управление и контроль через графы состояний; силён там, где важны циклы и явная логика переходов
- LlamaIndex ([github](https://github.com/run-llama/llama_index)) — про данные и RAG; силён в индексации и извлечении знаний.
- AutoGen - про мульти-агентное взаимодействие в формате разговора
- CrewAI ([github](https://github.com/crewAIInc/crewAI)) — про ролевые «команды» агентов с понятным разделением ролей

Иногда фреймворк не нужен вовсе. Anthropic в своем эссе [«Building Effective Agents»](https://www.anthropic.com/engineering/building-effective-agents) рекомендуют всегда начинать с простого - прямых вызовов модели - и добавлять сложность только тогда, когда она реально окупается



## Промышленные системы
Мир продакшена кардинально отличается от мира академии. Здесь на первый план выходят иные требования: оценка качества, грамотный мониторинг, обеспечение безопасности и оптимизация процессов.

### Оценка
Без измерения невозможна осмысленная итерация. Сложность в том, что агент недетерминирован, проходит много шагов и часто заслуживает «частичного зачёта», — простой accuracy здесь не работает

В арсенале 
— метрики успешности задач (довёл ли агент дело до конца) 
- LLM-as-a-judge (использовать другую модель как оценщика качества)
- специализированные бенчмарки:
    - [WebArena](https://arxiv.org/abs/2307.13854) (2023) для задач в вебе,
    - [GAIA](https://arxiv.org/abs/2311.12983) для ассистентов общего назначения,
    - [τ-bench (TauBench)](https://arxiv.org/abs/2406.12045) для взаимодействия с инструментами и пользователем в реалистичных доменах.

Отдельно стоит оценка RAG-компонентов — например, через [RAGAS](https://aclanthology.org/2024.eacl-demo.16), который измеряет качество извлечения и достоверность ответа

### Наблюдаемость и отладка
Нужно видеть, что происходит внутри многошагового процесса. Базовые инструменты — трейсинг всех вызовов (модели и инструментов), логирование шагов рассуждения и средства отладки недетерминированных сценариев, где один и тот же вход может приводить к разным траекториям

### Безопасность
Агент, способный выполнять реальные действия в среде — это реальная угроза и здесь ставки выше, чем у обычного чат-бота. Они могут быть преднамеренными — **prompt injection** и **jailbreak**: вредоносные инструкции, спрятанные во входных данных или на веб-странице, которые перехватывают поведение агента. Или непреднамеренными - забыли закрыть и агент при решении совей задачи воспользовался этим бэкдором. Для защиты использует те же принципы, как в кибербезопсаности: изоляция инструментов (агенты имеют доступ только к песочнице - изолированной копии продакшена), принцип минимальных прав (агент получает ровно те доступы, что нужны) и ограничение области действия агента (у агента набора интсрументов)

### Оптимизация

Чтобы агента можно было реально развернуть, он должен быть достаточно дешёвым и быстрым. Главные рычаги — контроль стоимости и латентности, кэширование (в том числе кэш промптов, чтобы не пересчитывать повторяющийся контекст), выбор модели под подзадачу (мелкая быстрая модель на простые шаги, крупная — только там, где нужна) и маршрутизация запросов между моделями разной мощности

### Обучение с подкреплением для агентов

Если в части 1 мы учили модель рассуждать с помощью RL, то теперь та же логика применяется к агентам целиком: вместо ручной настройки промптов и оркестрации — обучение агента через RL на взаимодействии со средой, end-to-end. Сюда же относятся самоулучшающиеся агенты (self improving agents) и множество открытых проблем (стабильность обучения, награды на длинном горизонте, перенос между задачами). 

Обзорная статья — [«The Landscape of Agentic Reinforcement Learning for LLMs: A Survey»](https://arxiv.org/abs/2509.02547) (2025)
